# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Kailaswadje/FlyRank-Internship_ML_Assignment_01_Week_01/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

**Task type: Scoring / Ranking** (not classification, not clustering).

Lane 4 — CTR / Engagement Opportunity Scoring — asks "which visible pages under-capture clicks or engagement relative to what their position and context predict?" That's inherently a relative, continuous question, not a yes/no one: a page isn't simply "underperforming" or "fine," it's underperforming *by some amount*, and the review queue needs pages ordered by that amount, not sorted into two buckets. The lane guide's own recommended methods confirm this framing — expected CTR by position tier, residual/gap analysis, and ranked scoring — none of which are classification or clustering methods.

I could later derive a binary "flag" from the score (e.g. "top 50 by gap"), but the underlying task producing that score is a scoring/ranking problem, and I want to keep that distinction honest rather than jumping straight to a yes/no label.

In [1]:
import pandas as pd

pd.set_option('display.width', 120)

df = pd.read_csv('/content/content_refresh_anonymized.csv')
print('starter dataset shape:', df.shape)

starter dataset shape: (30000, 44)


## 2. Target or proxy

**Proxy target: `ctr_gap`** — a page's observed CTR minus the *expected* CTR for its position tier (the tier's median CTR). A large negative gap means "clicking far less than similar-ranked pages typically do."

This is a **defined rule, not an observed future outcome** — I want to be explicit about that, per the lane guide's own warning against confusing a same-window proxy with a genuine future label. `ctr_gap` is calculated entirely from the current 90-day window; it tells me a page is behind its peers *right now*, not that it will keep declining or that fixing it will help. That's an honest limitation I'm carrying forward from Week 1, and it's something I'd want to strengthen later with an actual before/after or future-window check, not treat as settled.

In [2]:
filtered = df[(df['impressions_90d'] > 0) & (df['content_age_days'] >= 90)].drop_duplicates('content_id')

# Lane 4 slice: visible pages only (position 1-20, meaningful impression volume)
lane4 = filtered[(filtered['avg_position'] > 0) & (filtered['avg_position'] <= 20)
                  & (filtered['impressions_90d'] >= 500)].copy()

# Proxy target: ctr_gap = observed ctr - tier's median ctr
lane4['expected_ctr_for_tier'] = lane4.groupby('position_tier')['ctr'].transform('median')
lane4['ctr_gap'] = lane4['ctr'] - lane4['expected_ctr_for_tier']

print('ctr_gap summary (the proxy target):')
print(lane4['ctr_gap'].describe())

ctr_gap summary (the proxy target):
count    12023.000000
mean         0.099829
std          0.341475
min         -0.240000
25%         -0.110000
50%          0.000000
75%          0.190000
max          5.260000
Name: ctr_gap, dtype: float64


## 3. Success metric

**Metric: Precision@50** — of the top 50 pages ranked by worst `ctr_gap` (or, once modeled, by predicted opportunity score), how many would a reviewer agree are genuinely worth their time?

I'm picking this over a generic metric like RMSE or R² because the actual decision this supports is a fixed-capacity weekly review queue (see the Week 1 notebook) — a reviewer only opens a limited number of pages, so what matters is whether the *top* of the ranking is right, not how well the whole distribution of gaps is fit. Precision@50 mirrors exactly how the output gets used. "Good" would mean beating the flat single-threshold rule from Week 1 (which flagged 32.5% of pages — far more than any reviewer has time for) by a clear, validated margin, the same way the Week 1 starter pipeline showed a random forest reaching 0.740 precision@50 against a 0.240 baseline on a related task.

In [3]:
# No additional numbers needed here — precision@50 can't be computed yet without a
# real held-out label or reviewer judgment, which is Week 5+ work (training-honest-models,
# hunting-leakage-and-validating). This section states the metric and why it fits the decision.

## 4. The unit of analysis, as a real dataframe

**One row = one page** (`content_id`), for a single client, measured over the current 90-day window. Loaded and shown below, with the proxy target column attached.

In [4]:
cols = ['content_id', 'client_id', 'position_tier', 'avg_position', 'main_intent',
        'impressions_90d', 'ctr', 'expected_ctr_for_tier', 'ctr_gap']
print('Lane 4 unit-of-analysis slice:', lane4.shape, '-- one row per content_id')
lane4[cols].sort_values('ctr_gap').head(8)

Lane 4 unit-of-analysis slice: (12023, 46) -- one row per content_id


,content_id,client_id,position_tier,avg_position,main_intent,impressions_90d,ctr,expected_ctr_for_tier,ctr_gap
15334,content_bb6c9ae71383,client_6208ef0f77,page_1,3.0,informational,1475,0.0,0.24,-0.24
461,content_29edd4a67cf6,client_19581e27de,page_1,7.0,informational,1268,0.0,0.24,-0.24
6648,content_10d038b1695e,client_19581e27de,page_1,9.7,informational,599,0.0,0.24,-0.24
23745,content_c59a181ee3c0,client_4e07408562,page_1,8.3,informational,669,0.0,0.24,-0.24
18715,content_96472c1f2070,client_19581e27de,page_1,4.3,informational,714,0.0,0.24,-0.24
23776,content_e8ab88172458,client_a88a7902cb,page_1,6.4,informational,669,0.0,0.24,-0.24
7159,content_4b1303a5affe,client_7f2253d7e2,page_1,4.1,informational,1341,0.0,0.24,-0.24
24167,content_0895470266fd,client_6208ef0f77,page_1,7.7,informational,709,0.0,0.24,-0.24


## 5. Why ML beats a fixed rule here

A fixed rule like `ctr < 0.5` treats every page the same regardless of context — and Week 1 already showed that's too blunt (it flagged 32.5% of the dataset). Adjusting for position tier alone is better, but still not enough: **expected CTR shifts by search intent too, on top of position tier**, and the two interact rather than acting separately.

Below, median CTR by (position tier × intent) shows real movement within a single tier — e.g. on page_1, navigational queries median 0.325 CTR versus informational queries at 0.23 in the same tier. A single if-statement can encode one adjustment (tier) reasonably well, but stacking multiple interacting adjustments (tier, intent, and — eventually — content type, freshness, word count) by hand becomes unmanageable fast, and any hand-tuned combination is guesswork about which interactions matter. That's exactly the kind of multi-factor, interacting pattern a model can learn from data instead of guessing — which is why this earns real modeling rather than a longer chain of if/else rules.

In [5]:
# Does intent shift expected CTR on top of position tier? If yes, a tier-only rule
# still isn't enough -- multiple interacting factors argue for ML over a hand-written rule.
intent_tier_table = lane4.groupby(['position_tier', 'main_intent'])['ctr'].median().unstack().round(3)
print('Median CTR by position tier x intent:')
print(intent_tier_table)
print()

combo_counts = lane4.groupby(['position_tier', 'main_intent']).size()
print(f'(tier, intent) combinations with enough volume to trust (n>=30): '
      f'{(combo_counts>=30).sum()} out of {len(combo_counts)}')

Median CTR by position tier x intent:
main_intent    commercial  informational  navigational  transactional
position_tier                                                        
page_1               0.22           0.23         0.325           0.26
page_3_5             0.41           0.10           NaN           0.15
striking             0.17           0.17         0.390           0.18
top_3                0.20           0.16           NaN           0.32

(tier, intent) combinations with enough volume to trust (n>=30): 9 out of 14


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.